In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense

# 1. Load and preprocess data
df = pd.read_csv("Datasets/AutoEncoder/NASA Turbofan ENgine/NASA TurboFan Engine.csv")
df = df.fillna(method="ffill").fillna(0)

features = [f"sensor{i}" for i in range(1, 22)]
scaler = MinMaxScaler()
data = scaler.fit_transform(df[features])

# 2. Create time windows for each engine
window = 10

X, labels = [], []

for unit in df["unit"].unique():
    idx = df["unit"] == unit
    d = data[idx]
    y = df.loc[idx, "Fault_Label"].values

    X.extend([d[i:i+window] for i in range(len(d) - window + 1)])
    labels.extend([np.any(y[i:i+window] != 0) for i in range(len(d) - window + 1)])

X = np.array(X)
labels = np.array(labels)

X_train = X[labels == 0]

# 4. Build LSTM autoencoder
inp = Input(shape=(window, len(features)))
x = LSTM(32)(inp)
x = RepeatVector(window)(x)
x = LSTM(32, return_sequences=True)(x)
out = TimeDistributed(Dense(len(features)))(x)

model = Model(inp, out)
model.compile(optimizer="adam", loss="mse")
model.fit(X_train, X_train, epochs=20, batch_size=32, validation_split=0.1)

# 5. Detect faults using reconstruction error
reconstructed = model.predict(X)
error = np.mean((X - reconstructed) ** 2, axis=(1, 2))
threshold = np.percentile(error[labels == 0], 90)
predictions = (error > threshold).astype(int)

# 6. Display results
print("Threshold:", threshold)
print("Actual faults:", labels.sum())
print("Detected faults:", predictions.sum())
print("Accuracy:", np.mean(predictions == labels))

from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(labels, predictions))
print(classification_report(labels, predictions))

# print(df["Fault_Label"].value_counts())

# print("Healthy windows:", np.sum(labels == 0))
# print("Faulty windows:", np.sum(labels == 1))

# print("Healthy error:", np.mean(error[labels == 0]))
# print("Faulty error:", np.mean(error[labels == 1]))

# print("Healthy 95th percentile:", np.percentile(error[labels == 0], 95))
# print("Faulty 50th percentile:", np.percentile(error[labels == 1], 50))

C:\Users\kmchw\AppData\Local\Temp\ipykernel_13224\1474536832.py:9: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill").fillna(0)


Epoch 1/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - loss: 0.0240 - val_loss: 0.0039
Epoch 2/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.0041 - val_loss: 0.0037
Epoch 3/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 0.0039 - val_loss: 0.0037
Epoch 4/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.0038 - val_loss: 0.0036
Epoch 5/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 0.0038 - val_loss: 0.0036
Epoch 6/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.0038 - val_loss: 0.0036
Epoch 7/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.0037 - val_loss: 0.0035
Epoch 8/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 0.0037 - val_loss: 0.0035
Epoch 9/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 0.0037 - val_loss: 0.0034
Epoch 10/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0036 - val_loss: 0.0034
Epoch 11/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.0036 - val_loss: 0.0034
Epoch 12/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 9m

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.layers import Input, Dense, LSTM, RepeatVector, TimeDistributed
from tensorflow.keras.models import Model

df = pd.read_csv("Datasets/AutoEncoder/NASA Turbofan ENgine/NASA Turbofan Engine.csv")

df.head()

features = [f"sensor{i}" for i in range(1, 22)]

scaler = MinMaxScaler()
data = scaler.fit_transform(df[features])

window = 10

X, labels = [], []

for unit in df["unit"].unique():
    ind = df["unit"] == unit
    d = data[ind]
    y = df.loc[ind, "Fault_Label"].values

    X.extend([d[i : i + window] for i in range(len(d) - window + 1)])
    labels.extend([np.any(y[i : i + window] != 0) for i in range(len(d) - window + 1)])

X = np.array(X)
labels = np.array(labels)

X_train = X[labels == 0]

inp = Input(shape=(window, len(features)))
en = LSTM(32, activation='relu')(inp)
en = RepeatVector(window)(en)
en = LSTM(32, activation='relu', return_sequences=True)(en)
out = TimeDistributed(Dense(len(features)))(en)

model = Model(inp, out)

model.compile(optimizer='adam', loss='mse')

model.fit(X_train, X_train, epochs=20, batch_size=32, validation_split=0.1)

reconstruct = model.predict(X, verbose=0)
error = np.mean((X - reconstruct) ** 2, axis=(1, 2))

threshold = np.percentile(error[labels == 0], 85)
preds = error > threshold

print("Threshold: ", threshold)
print("Actual Faults: ", labels.sum())
print("Predicted Faults: ", preds.sum())
print("Accuracy: ", np.mean(preds == labels))

print("Confusion Matrix: ", confusion_matrix(labels, preds))
print("Classification Report: ", classification_report(labels, preds))

Epoch 1/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - loss: 0.0374 - val_loss: 0.0038
Epoch 2/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.0040 - val_loss: 0.0037
Epoch 3/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.0039 - val_loss: 0.0037
Epoch 4/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0039 - val_loss: 0.0037
Epoch 5/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0038 - val_loss: 0.0036
Epoch 6/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0038 - val_loss: 0.0036
Epoch 7/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0038 - val_loss: 0.0036
Epoch 8/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0037 - val_loss: 0.0036
Epoch 9/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0038 - val_loss: 0.0035
Epoch 10/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0037 - val_loss: 0.0035
Epoch 11/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0037 - val_loss: 0.0035
Epoch 12/20
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/st